In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.covariance import LedoitWolf

## 1. Loading the Data

In [2]:
with open("hw3_input.pickle", "rb") as f:
    data = pickle.load(f)

In [3]:
price = data["price"] # Txn
tri = data["tri"] # Txn
volume = data["volume"] # Txn
mtbv = data["mtbv"] # Txn
cap = data["cap"] # Txn
tcost = data["tcost"] # Txn
rec = data["rec"] # Txn
isactivenow = data["isactive"] # Txn

allstocks = price.columns.to_numpy() # 1xn
myday = price.index.to_numpy() # Tx1

# T and n
T, n = price.shape
print(f"T = {T}, n = {n}")

T = 1521, n = 411


## 2. Risk Model

In [4]:
tri_clean = tri.copy()
tri_clean = tri_clean.apply(pd.to_numeric, errors="coerce")

# Compute arithmetic daily returns manually: R_t = TRI_t / TRI_{t-1} - 1 since pct_change is causing issues
previous_tri = tri_clean.shift(1)
returns = (tri_clean / previous_tri) - 1
returns = returns.replace([np.inf, -np.inf], np.nan)
returns = returns.fillna(0)

all_dates = returns.index

In [5]:
# Collect (year, month) pairs, set up vectors and lists for later
year_month_list = []
for dt in all_dates:
    ym = (dt.year, dt.month)
    if ym not in year_month_list:
        year_month_list.append(ym)

# shrinkage intensities (alpha) for each month
shrink_list = [] # months x 1 vector

# first trading date per month
month_ref_dates = [] # one date per month, same order as shrink_list

# covariance matrices per month
cov_by_month = {} # key: ref_date, value: covariance DataFrame

In [6]:
def get_month_reference_date(year, month, date_index):

    first_calendar_day = pd.Timestamp(year=year, month=month, day=1)
    
    # find all dates <= first_calendar_day
    mask = date_index <= first_calendar_day
    if not mask.any():
        return None
    
    # reference date is the max of those
    ref_date = date_index[mask].max()
    return ref_date

In [7]:
for (year, month) in year_month_list:
    # get reference date for this month
    ref_date = get_month_reference_date(year, month, all_dates)
    if ref_date is None:
        continue
    
    # find the position of ref_date in the index
    try:
        ref_pos = all_dates.get_loc(ref_date)
    except KeyError:
        continue
    
    # check if we have at least 252 days before this date
    if ref_pos < 252:
        continue
    
    # define the 252-day lookback window
    start_pos = ref_pos - 252
    end_pos = ref_pos  # slice is [start_pos, end_pos) in iloc
    
    returns_window = returns.iloc[start_pos:end_pos]  # shape: (252, n)
    
    # determine active stocks at the reference date
    active_row = isactivenow.loc[ref_date]
    active_mask = (active_row == 1)
    
    # list of active stocks
    active_stocks = list(isactivenow.columns[active_mask])
    
    if len(active_stocks) == 0:
        continue
    
    # restrict returns to active universe
    R = returns_window[active_stocks] # DataFrame (252 x n_active)
    
    # fill NaNs
    R_filled = R.fillna(0)
    
    # Ledoit-Wolf shrinkage
    X = R_filled.values # shape: (252, n_active)
    lw = LedoitWolf(assume_centered=False)
    
    # fit covariance matrix
    lw.fit(X)
    cov_matrix = lw.covariance_ # numpy array (n_active x n_active)
    alpha = float(lw.shrinkage_) # shrinkage intensity
    
    # if intensity is negative, set to 0
    if alpha < 0:
        alpha = 0.0
    
    # store shrinkage intensity
    shrink_list.append(alpha)
    month_ref_dates.append(ref_date)
    
    # turn covariance into a DataFrame for convenience later
    cov_df = pd.DataFrame(
        cov_matrix,
        index=active_stocks,
        columns=active_stocks
    )
    cov_by_month[ref_date] = cov_df


In [8]:
shrink = np.array(shrink_list).reshape(-1, 1)

print("Number of months with risk model:", shrink.shape[0])

#shrink

Number of months with risk model: 59


## 3. Alphas

In [9]:
def process_alpha(alpha_raw, isactivenow, winsor_limit=3.0):
    """
    For active stocks, demean, standardize, winsorize.
    For inactive stocks, we set the alpha to 0.
    """
    alpha = alpha_raw.copy().astype(float)
    
    for date in alpha.index:
        # which stocks are active on this date?
        active_mask = (isactivenow.loc[date] == 1)
        if active_mask.sum() == 0:
            # no active stocks → set whole row to 0
            alpha.loc[date, :] = 0.0
            continue
        
        # Values for active stocks
        vals = alpha.loc[date, active_mask]
        
        mean_val = vals.mean()
        std_val = vals.std()
        
        # If std is 0 or NaN, avoid division; set row to 0
        if (std_val is None) or (std_val == 0) or np.isnan(std_val):
            alpha.loc[date, :] = 0.0
            continue
        
        # z-score: (x - mean) / std
        z = (vals - mean_val) / std_val
        
        # winsorize: cap at +/- winsor_limit
        z = z.clip(lower=-winsor_limit, upper=winsor_limit)
        
        # Put back into alpha
        alpha.loc[date, active_mask] = z
        
        # For inactive stocks, set to 0
        alpha.loc[date, ~active_mask] = 0.0
    
    # Replace any remaining NaNs with 0
    alpha = alpha.fillna(0.0)
    return alpha

### SHORT-TERM CONTRARIAN

In [10]:
# Parameters
K_REV = 10  # lookback window for reversal (e.g., 10 days)

# Initialize alpharev with zeros
alpharev_raw = pd.DataFrame(0.0, index=all_dates, columns=allstocks)

# Triangular weights: [10, 9, ..., 1]
weights = list(range(1, K_REV + 1))  # [1,2,...,10]
weights = weights[::-1]              # [10,9,...,1]

for t in range(K_REV, T):
    # Take the last K_REV days of returns: rows t-K_REV ... t-1
    window = returns.iloc[t-K_REV:t]   # shape (K_REV x n)
    
    # Weighted sum across time
    weighted_sum = pd.Series(0.0, index=allstocks)
    
    for k in range(K_REV):
        day_returns = window.iloc[k]      # this is a Series for one day
        weight = weights[k]
        weighted_sum += weight * day_returns
    
    # Contrarian: negative sign
    alpharev_raw.iloc[t] = -weighted_sum

# For first K_REV days, we leave alpharev_raw as 0 (no signal yet)

alpharev = process_alpha(alpharev_raw, isactivenow)
alpharev

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.053452,-0.053452,-0.053452,-0.053452,0.0,-0.053452,-0.053452,-0.053452,-0.053452,0.0,...,0.0,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452
2025-07-29,-0.053452,-0.053452,-0.053452,-0.053452,0.0,-0.053452,-0.053452,-0.053452,-0.053452,0.0,...,0.0,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452,-0.053452
2025-07-30,0.414744,0.142090,0.084267,-0.467651,0.0,-1.235651,-0.629211,0.274938,0.062711,0.0,...,0.0,0.768795,-0.231388,0.008378,0.230157,-0.503466,0.062735,-0.224285,-0.027932,-0.101421


### SHORT-TERM PROCYCLICAL

In [11]:
H_REC = 20  # lookback window for recommendation revisions

# Raw recommendation revision: rec_t - rec_{t-20}
alpharec_raw = rec - rec.shift(H_REC)

# Replace NaNs (early days) with 0
alpharec_raw = alpharec_raw.fillna(0)

# Process it
alpharec = process_alpha(alpharec_raw, isactivenow)
alpharec

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.101798,-0.101798,-0.101798,-0.101798,0.0,1.877615,-1.091505,-0.101798,-0.101798,0.0,...,0.0,-1.091505,-0.101798,-1.091505,0.887908,0.887908,-0.101798,-0.101798,0.887908,-2.081212
2025-07-29,-0.107566,-0.107566,-0.107566,-0.107566,0.0,1.927464,-1.125081,-0.107566,-0.107566,0.0,...,0.0,-1.125081,-0.107566,-0.107566,0.909949,0.909949,-0.107566,-0.107566,0.909949,-2.142595
2025-07-30,-0.133384,-0.133384,-0.133384,-0.133384,0.0,1.935547,-1.167849,-0.133384,-0.133384,0.0,...,0.0,-1.167849,-0.133384,-0.133384,0.901081,0.901081,-0.133384,-0.133384,0.901081,-2.202314


### LONG-TERM CONTRARIAN

In [15]:
alphaval_raw = -mtbv.copy().astype(float)

# Handle NaNs
alphaval_raw = alphaval_raw.fillna(0)

# Process it
alphaval = process_alpha(alphaval_raw, isactivenow)
alphaval

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-29,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-30,0.205700,0.168883,0.622919,0.077622,0.0,-1.792137,0.454675,0.025840,0.285338,0.0,...,0.0,-0.208879,-0.713995,-2.213715,0.712331,0.531899,-0.550509,-0.147500,0.478032,0.600817


### LONG-TERM PROCYCLICAL

In [16]:
SKIP_1M = 21   # days to skip (most recent month)
LOOKBACK_12M = 252  # days for 12-month window

# TRI at t-21 and t-252
tri_1m_ago = tri_clean.shift(SKIP_1M)
tri_12m_ago = tri_clean.shift(LOOKBACK_12M)

alphamom_raw = (tri_1m_ago / tri_12m_ago) - 1

# Clean NaNs and infinite values
alphamom_raw = alphamom_raw.replace([np.inf, -np.inf], np.nan)
alphamom_raw = alphamom_raw.fillna(0.0)

# Process it
alphamom = process_alpha(alphamom_raw, isactivenow)
alphamom

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.051704,-0.098601,-0.078957,0.122306,0.0,0.109639,0.274946,0.083713,-0.159285,0.0,...,0.0,0.651695,0.646750,0.279946,-0.093449,-0.063485,-0.046062,0.040743,0.164973,-0.034679
2025-07-29,-0.173063,0.213880,0.005538,-0.250725,0.0,0.011196,-0.109731,0.097634,0.286369,0.0,...,0.0,0.259841,-0.816989,-1.449810,0.966092,0.001659,-0.058754,-0.401767,-0.319902,0.227780
2025-07-30,0.042158,0.042099,0.160982,-0.006243,0.0,0.055492,-0.010200,0.063182,-0.001575,0.0,...,0.0,0.031610,-0.052049,-0.034930,0.239062,0.121451,0.118930,0.006784,-0.061160,0.002558


### BLEND

In [17]:
alphablend_raw = (
    0.50 * alpharev +
    0.25 * alpharec +
    0.15 * alphaval +
    0.10 * alphamom
)

alphablend = process_alpha(alphablend_raw, isactivenow)
alphablend

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.002434,-0.074513,0.171357,-0.045094,0.0,0.701357,-0.660634,-0.085881,-0.034493,0.0,...,0.0,-0.877802,-0.277883,-2.063390,1.080859,0.996394,-0.434744,-0.192185,1.048143,-1.559793
2025-07-29,-0.093188,0.026020,0.196117,-0.190205,0.0,0.723635,-0.850360,-0.092940,0.114908,0.0,...,0.0,-1.075555,-0.820849,-1.857476,1.503866,1.059884,-0.460120,-0.365958,0.915246,-1.564473
2025-07-30,0.390377,0.108606,0.210072,-0.533761,0.0,-0.813948,-1.096357,0.202114,0.055792,0.0,...,0.0,0.102692,-0.544073,-0.749386,0.910811,0.105082,-0.169288,-0.356546,0.525043,-1.039795
